### Checking available pre-loaded datasets inside databricks

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/"))

### Checking for available pre loaded files in NYC Taxi trip data

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/"))

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/"))

path,name,size,modificationTime
dbfs:/databricks-datasets/nyctaxi/readme_nyctaxi.txt,readme_nyctaxi.txt,916,1596568072000
dbfs:/databricks-datasets/nyctaxi/reference/,reference/,0,1779428898885
dbfs:/databricks-datasets/nyctaxi/sample/,sample/,0,1779428898885
dbfs:/databricks-datasets/nyctaxi/tables/,tables/,0,1779428898885
dbfs:/databricks-datasets/nyctaxi/taxizone/,taxizone/,0,1779428898885
dbfs:/databricks-datasets/nyctaxi/tripdata/,tripdata/,0,1779428898885


### Getting column name for schema

In [0]:
path = "dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-01.csv.gz"

column_names = spark.read.option("header", "true").csv(path).limit(0).columns

print(column_names)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']


### Created Schema to load dataset faster

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("vendor_id", StringType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])

df = (spark.read
      .format("csv")
      .option("header", "true")
      .schema(schema) 
      .load(path)
     )

display(df.limit(100)) 

### Bronze Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

### Added metadata columns inside bronze dataset

In [0]:
from pyspark.sql.functions import *

bronze_df = df \
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    ) \
    .withColumn(
        "load_date",
        current_date()
    ) \
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

(bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.nyc_taxi_raw"))

In [0]:
df.select("_metadata.*").display()

In [0]:
%sql
SELECT *
FROM bronze.nyc_taxi_raw
LIMIT 10;